# Length-Stress Test — Data Prep

Replicates the original Korean-language prep notebook's method
(paragraph-boundary truncation ladder + per-question `answerable` flags) on
English data. This is the first notebook in the experiment sequence (`00_pipeline_test`
is a standalone sanity check, not a pipeline step) — it builds the
`novel`-genre corpus that `02_length_stress_test.ipynb` (this data's own
length-stress experiment) and `03_qa_baseline_3conditions.ipynb` (as the
`novel` genre) consume downstream.

**Source**: `deepmind/narrativeqa`, document id
`11ac9bf7f55ee9d40df3c0ba117c6914dd6ed9e6` — *A Thief in the Night* by
E. W. Hornung (Project Gutenberg #2098, public domain), ~63,700 words after
stripping the Project Gutenberg header/footer boilerplate. Selected by
scanning ~80 narrativeqa documents for the one with the most
**verbatim-findable** answers (21 of its 30 questions) — see §2 for why that
matters.

Unlike the old KorQuAD-based corpus (which needed 26 short documents
concatenated together to reach a usable length + question count),
narrativeqa documents are already full books — one is already far longer
than our 10,000-word max target, so no concatenation is needed here.

Truncation happens **only at paragraph boundaries** — cutting mid-document
could produce a malformed paragraph or a cut-off sentence in
`plain_text_to_paragraphs`, so we stop just before the target word count is
exceeded (actual per-version word counts are therefore approximate, not
exact — measured below and used as the actual basis for `answerable`).

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import pandas as pd
from datasets import load_dataset

from src.pipeline.chuncking import plain_text_to_paragraphs

project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the document + strip Project Gutenberg boilerplate

Project Gutenberg texts wrap the actual content between
`*** START OF THIS PROJECT GUTENBERG EBOOK ... ***` and
`*** END OF THIS PROJECT GUTENBERG EBOOK ... ***` markers — everything
outside that (license text, transcriber notes) isn't part of the actual
narrative and would pollute both the corpus and any answer-position
calculations.

In [2]:
DOC_ID = "11ac9bf7f55ee9d40df3c0ba117c6914dd6ed9e6"

ds = load_dataset("deepmind/narrativeqa", split="train")
doc_rows = ds.filter(lambda ex: ex["document"]["id"] == DOC_ID)

raw_text = doc_rows[0]["document"]["text"]
start_idx = raw_text.find("*** START")
end_idx = raw_text.find("*** END")
content_start = raw_text.find("\n", start_idx)
full_text = raw_text[content_start:end_idx].strip()

print(f"raw text: {len(raw_text)} chars")
print(f"after stripping boilerplate: {len(full_text)} chars, {len(full_text.split())} words")
print(f"questions for this document: {len(doc_rows)}")

Filter:   0%|          | 0/32747 [00:00<?, ? examples/s]

Filter:   3%|▎         | 1000/32747 [00:01<00:45, 698.89 examples/s]

Filter:   6%|▌         | 2000/32747 [00:01<00:26, 1157.82 examples/s]

Filter:   9%|▉         | 3000/32747 [00:02<00:26, 1107.05 examples/s]

Filter:  12%|█▏        | 4000/32747 [00:03<00:27, 1044.10 examples/s]

...

Filter:  95%|█████████▍| 31000/32747 [00:57<00:04, 384.06 examples/s]

Filter:  98%|█████████▊| 32000/32747 [01:00<00:02, 371.03 examples/s]

Filter: 100%|██████████| 32747/32747 [01:02<00:00, 361.00 examples/s]

Filter: 100%|██████████| 32747/32747 [01:02<00:00, 521.60 examples/s]

raw text: 359935 chars
after stripping boilerplate: 340702 chars, 63658 words
questions for this document: 30


## 2. Filter to verbatim-findable answers + compute position

narrativeqa answers are free-form (crowd-written), unlike the old KorQuAD
corpus where each answer was an explicit extractive span with a known
position. Most narrativeqa answers are paraphrases (confirmed empirically:
only ~39% of a 200-question sample matched verbatim), so we can only
compute a reliable evidence position for the subset where at least one of
the (usually 2) answer variants appears verbatim in the text — the same
`text.find(answer)` + word-position approach `build_length_stress_corpus.py`
used for KorQuAD. Questions with no verbatim-findable answer are dropped —
there's no way to assign them a reliable evidence position, and an
unpositioned question can't be used for a length/position-sensitive
analysis anyway.

In [3]:
def word_index(text: str, char_pos: int) -> int:
    """1-indexed word position: word count before char_pos, plus 1."""
    return len(text[:char_pos].split()) + 1


def find_verbatim_answer(answers: list[dict], text: str) -> tuple[str, int] | None:
    """First answer variant that appears verbatim in text, with its char position."""
    for a in answers:
        pos = text.find(a["text"])
        if pos >= 0:
            return a["text"], pos
    return None


questions = []
for i, row in enumerate(doc_rows):
    found = find_verbatim_answer(row["answers"], full_text)
    if found is None:
        continue
    answer_text, char_pos = found
    questions.append(
        {
            "question_id": i,
            "question": row["question"]["text"],
            "answer": answer_text,
            "evidence_char_pos": char_pos,
            "evidence_word_pos": word_index(full_text, char_pos),
            "type": "narrativeqa_extractive_subset",
        }
    )

questions_df = pd.DataFrame(questions)
print(f"verbatim-findable questions: {len(questions_df)} / {len(doc_rows)}")
questions_df.head(3)

verbatim-findable questions: 21 / 30


,question_id,question,answer,evidence_char_pos,evidence_word_pos,type
0,0,What kind of crime do Raffle and Bunnys commit?,Burglary,186004,34765,narrativeqa_extractive_subset
1,1,What was Raffles famous for?,cricket,6770,1270,narrativeqa_extractive_subset
2,2,What does Raffles say he was asked about for?,cricket,6770,1270,narrativeqa_extractive_subset


## 3. Truncate at paragraph boundaries

Same algorithm as the original Korean-language notebook: use
`paragraphs[i+1].char_offset` (the real position in the original text where
the next paragraph starts) as the cut point — not
`char_offset + len(paragraph.text)`, since `plain_text_to_paragraphs`
normalizes internal whitespace/newlines, which could introduce drift if we
computed the boundary from the normalized paragraph text instead of the
original document.

In [4]:
TARGET_WORD_COUNTS = [1000, 2000, 3000, 6000, 10000]

paragraphs = plain_text_to_paragraphs(full_text)
print(f"total paragraphs: {len(paragraphs)}")


def truncate_to_word_count(paragraphs: list, full_text: str, target_words: int) -> str:
    """Cut at the nearest paragraph boundary without exceeding target_words."""
    cumulative_words = 0
    cut_char_pos = 0
    for i, p in enumerate(paragraphs):
        p_words = len(p.text.split())
        if cumulative_words > 0 and cumulative_words + p_words > target_words:
            break
        cumulative_words += p_words
        next_offset = paragraphs[i + 1].char_offset if i + 1 < len(paragraphs) else len(full_text)
        cut_char_pos = next_offset
    return full_text[:cut_char_pos]


versions = {target: truncate_to_word_count(paragraphs, full_text, target) for target in TARGET_WORD_COUNTS}
for target, version_text in versions.items():
    print(f"target {target:>6}: actual {len(version_text.split())} words")

total paragraphs: 1255
target   1000: actual 946 words
target   2000: actual 1983 words
target   3000: actual 2946 words
target   6000: actual 5991 words
target  10000: actual 9945 words


## 4. Save versions + compute `answerable` flags

A question is `answerable` for a given version if its `evidence_word_pos`
falls within that version's actual (measured, not target) word count.

In [5]:
OUT_DIR = Path("data/processed/narrativeqa_length_stress")
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR = Path("data/sample")
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

full_text_path = OUT_DIR / "length_stress_corpus.txt"
full_text_path.write_text(full_text, encoding="utf-8")

actual_word_counts = {}
for target, version_text in versions.items():
    version_path = OUT_DIR / f"length_stress_v{target}.txt"
    version_path.write_text(version_text, encoding="utf-8")
    actual_word_counts[target] = len(version_text.split())
    print(f"saved: {version_path} ({actual_word_counts[target]} words)")

for target in TARGET_WORD_COUNTS:
    col = f"answerable_v{target}"
    questions_df[col] = questions_df["evidence_word_pos"] <= actual_word_counts[target]

QUESTIONS_PATH = SAMPLE_DIR / "narrativeqa_length_stress.csv"
questions_df.to_csv(QUESTIONS_PATH, index=False, encoding="utf-8-sig")
print(f"\nsaved: {QUESTIONS_PATH}")
questions_df[["question_id", "question", "evidence_word_pos"] + [f"answerable_v{t}" for t in TARGET_WORD_COUNTS]]

saved: data/processed/narrativeqa_length_stress/length_stress_v1000.txt (946 words)
saved: data/processed/narrativeqa_length_stress/length_stress_v2000.txt (1983 words)
saved: data/processed/narrativeqa_length_stress/length_stress_v3000.txt (2946 words)
saved: data/processed/narrativeqa_length_stress/length_stress_v6000.txt (5991 words)
saved: data/processed/narrativeqa_length_stress/length_stress_v10000.txt (9945 words)

saved: data/sample/narrativeqa_length_stress.csv


,question_id,question,evidence_word_pos,answerable_v1000,answerable_v2000,answerable_v3000,answerable_v6000,answerable_v10000
0,0,What kind of crime do Raffle and Bunnys commit?,34765,False,False,False,False,False
1,1,What was Raffles famous for?,1270,False,True,True,True,True
2,2,What does Raffles say he was asked about for?,1270,False,True,True,True,True
3,4,Raffles jumps overboard on the ocean voyage he...,63080,False,False,False,False,False
4,5,Where was Bunny before being summoned to the h...,7405,False,False,False,False,True
5,6,Who is the rich invalid?,590,True,True,True,True,True
6,7,Where was Bunny when he summoned Raffles?,2170,False,False,True,True,True
7,9,Who dies in battle?,9,True,True,True,True,True
8,11,Who summoned Bunny after he was released from ...,9,True,True,True,True,True
9,13,Which character is wounded in the war?,590,True,True,True,True,True


## 5. Answerable question counts per version

In [6]:
summary = pd.DataFrame(
    {
        "target_words": TARGET_WORD_COUNTS,
        "actual_words": [actual_word_counts[t] for t in TARGET_WORD_COUNTS],
        "answerable_questions": [questions_df[f"answerable_v{t}"].sum() for t in TARGET_WORD_COUNTS],
    }
)
summary["answerable_ratio"] = (summary["answerable_questions"] / len(questions_df)).round(2)
summary

,target_words,actual_words,answerable_questions,answerable_ratio
0,1000,946,11,0.52
1,2000,1983,13,0.62
2,3000,2946,14,0.67
3,6000,5991,14,0.67
4,10000,9945,16,0.76


## Summary

- 5 truncated versions
  (`data/processed/narrativeqa_length_stress/length_stress_v{1000,2000,3000,6000,10000}.txt`)
  and a question CSV with per-version `answerable_vN` flags
  (`data/sample/narrativeqa_length_stress.csv`).
- Started from 30 narrativeqa questions for this document; kept the 21 whose
  answer is verbatim-findable (the rest have no reliable evidence position
  and were dropped — see §2).
- Consumed by two downstream notebooks: `02_length_stress_test.ipynb` (this
  notebook's own length-stress QA experiment) and
  `03_qa_baseline_3conditions.ipynb` (as the `novel` genre corpus).
  `00_pipeline_test.ipynb` exercises the same document but reads its own
  committed copy (`data/sample/pipeline_test_sample.txt`), so it does not
  depend on this notebook having been run.
- Next (`02_length_stress_test.ipynb`): run closed_book vs. full_context QA
  across the 5 versions, scoring **only** questions where
  `answerable_vN=True` for that version — an unanswerable question isn't
  "wrong," it's out of scope for that version's budget, and scoring it as
  wrong would conflate "couldn't fit in the budget" with "read it and still
  missed it." Answers are open-ended (not multiple-choice), so scoring needs
  normalized partial-match, not the 4-option parser from
  `03_qa_baseline_3conditions.ipynb`.